In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from pandas.api.types import (
    is_numeric_dtype,
    is_object_dtype,
    is_string_dtype
)


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_J.xpt"
csv_file = DATA_DIR / "SMQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 SMQ_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD057",
    "SMQ078",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ621",
    "SMD630",
    "SMQ661",
    "SMQ665A",
    "SMQ665B",
    "SMQ665C",
    "SMQ665D",
    "SMQ670",
    "SMQ848",
    "SMQ852Q",
    "SMQ852U",
    "SMQ890",
    "SMQ895",
    "SMQ900",
    "SMQ905",
    "SMQ910",
    "SMQ915",
    "SMAQUEX2"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQ_J columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6724:

    raise ValueError(
        f"Unexpected SMQ_J row count: "
        f"{len(df):,}"
    )


if len(df.columns) != 37:

    raise ValueError(
        f"Unexpected SMQ_J column count: "
        f"{len(df.columns)}"
    )


print(
    "PASS: Structure = 6,724 rows x 37 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# SMQ_J is a MIXED-TYPE questionnaire file.
#
# CHARACTER:
#
#     SMDUPCA
#         12-character cigarette UPC.
#
#     SMD100BR
#         cigarette brand/sub-brand text.
#
# GENUINE DECIMAL:
#
#     SMD100NI
#         FTC nicotine content.
#
# ALL OTHER numeric fields contain whole-number
# values/codes.
#
# IMPORTANT:
#
# Do NOT convert SMD100NI to Int64.
#
# Do NOT convert SMDUPCA to numeric.
#

CHARACTER_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


DECIMAL_COLUMNS = [
    "SMD100NI"
]


NUMERIC_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in CHARACTER_COLUMNS
]


INTEGER_COLUMNS = [
    col
    for col in NUMERIC_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(CHARACTER_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly 2 character variables."
    )


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 genuine decimal variable."
    )


if len(NUMERIC_COLUMNS) != 35:

    raise ValueError(
        "Expected exactly 35 numeric variables."
    )


if len(INTEGER_COLUMNS) != 34:

    raise ValueError(
        "Expected exactly 34 whole-number numeric fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Character fields:",
    CHARACTER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code numeric fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. VERIFY SOURCE DTYPES
# ============================================================

#
# IMPORTANT:
#
# Pandas versions may load SAS character fields as:
#
#     object
#     string
#     str
#     StringDtype
#
# Therefore:
#
#     df[col].dtype == object
#
# is TOO STRICT.
#
# The critical requirement is simply that SMDUPCA and
# SMD100BR are NOT numeric.
#

actual_numeric = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if actual_numeric != NUMERIC_COLUMNS:

    raise ValueError(
        "Unexpected numeric-column structure.\n\n"
        f"Expected:\n{NUMERIC_COLUMNS}\n\n"
        f"Actual:\n{actual_numeric}"
    )


print("\n--- SOURCE DTYPE CHECK ---")


for col in CHARACTER_COLUMNS:

    dtype = df[col].dtype

    print(
        f"{col}: dtype = {dtype}"
    )

    if is_numeric_dtype(df[col]):

        raise ValueError(
            f"STOP: {col} loaded as numeric dtype "
            f"({dtype}).\n"
            "Character fidelity may be at risk."
        )

    if not (
        is_object_dtype(dtype)
        or
        is_string_dtype(dtype)
    ):

        raise ValueError(
            f"{col} loaded with unexpected dtype: "
            f"{dtype}"
        )


print(
    "PASS: Character fields loaded as "
    "non-numeric text-compatible columns."
)


# ============================================================
# 5A. NORMALIZE CHARACTER COLUMNS SAFELY
# ============================================================

#
# Explicitly keep character fields as pandas StringDtype.
#
# This protects leading zeros in SMDUPCA.
#
# Example:
#
#     026100005738
#
# remains:
#
#     026100005738
#

for col in CHARACTER_COLUMNS:

    df[col] = (
        df[col]
        .astype("string")
    )


print(
    "PASS: SMDUPCA and SMD100BR explicitly "
    "retained as text."
)


# ============================================================
# 6. KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER blindly replace this across the whole dataframe.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[NUMERIC_COLUMNS]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before repair:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated attached SMQ_J pattern:
#
#     SMD030       62
#     SMD641       38
#     SMD100FL     15
#     SMD100MN    553
#     SMQ852Q       3
#     SMQ895     1711
#     SMQ905      844
#     SMQ915      720
#
# TOTAL = 3,946
#

EXPECTED_TINY_COUNTS = {
    "SMD030": 62,
    "SMD641": 38,
    "SMD100FL": 15,
    "SMD100MN": 553,
    "SMQ852Q": 3,
    "SMQ895": 1711,
    "SMQ905": 844,
    "SMQ915": 720
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: SMQ_J tiny-value pattern differs "
        "from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3946:

    raise ValueError(
        f"Unexpected total tiny count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Exact 3,946-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    actual_count = int(
        mask.sum()
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col} tiny count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )

    df.loc[
        mask,
        col
    ] = 0

    correction_counts[col] = actual_count


print("\n--- VALIDATED ZERO RESTORATION ---")


for col, count in correction_counts.items():

    print(
        f"{col}:",
        f"{count:,}"
    )


if (
    sum(
        correction_counts.values()
    )
    != 3946
):

    raise ValueError(
        "Total zero-restoration count incorrect."
    )


print(
    "PASS: 3,946 validated zeros restored."
)


# ============================================================
# 9. EXPECTED CORRECTED SOURCE
# ============================================================

#
# The ONLY permitted numeric changes are the validated
# tiny-value -> zero restorations.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny XPORT values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after repair."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE converting integer variables.
#
# np.round() is used ONLY as a comparison test.
#
# It DOES NOT alter the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_counts = {}


for col in NUMERIC_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_counts[col] = fractional_count


#
# SMD100NI is the only genuine decimal field.
#

EXPECTED_FRACTIONAL_COUNTS = {
    "SMD100NI": 553
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-variable pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: SMD100NI is the only genuine "
    "decimal variable."
)


# ============================================================
# 12. VALIDATE SMD100NI DECIMAL FIELD
# ============================================================

nicotine = (
    df["SMD100NI"]
    .dropna()
)


print("\n--- SMD100NI DECIMAL FIDELITY ---")

print(
    "Populated:",
    len(nicotine)
)

print(
    "Minimum:",
    nicotine.min()
)

print(
    "Maximum:",
    nicotine.max()
)

print(
    "Fractional observations:",
    fractional_counts["SMD100NI"]
)


if len(nicotine) != 695:

    raise ValueError(
        "Unexpected SMD100NI nonmissing count."
    )


if not np.isclose(
    nicotine.min(),
    0.5,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected SMD100NI minimum."
    )


if not np.isclose(
    nicotine.max(),
    2.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected SMD100NI maximum."
    )


if int(
    df["SMD100NI"]
    .isna()
    .sum()
) != 6029:

    raise ValueError(
        "Unexpected SMD100NI missing count."
    )


print(
    "PASS: SMD100NI genuine decimal values preserved."
)


# ============================================================
# 13. CHARACTER FIELD FIDELITY CHECK
# ============================================================

print("\n--- CHARACTER FIELD CHECK ---")


upc = (
    df["SMDUPCA"]
    .fillna("")
    .astype("string")
)


brand = (
    df["SMD100BR"]
    .fillna("")
    .astype("string")
)


upc_blank_count = int(
    upc.eq("")
    .sum()
)


upc_no_match_count = int(
    upc.eq("* NO MATCH *")
    .sum()
)


upc_recorded_count = int(
    upc.str.fullmatch(
        r"\d{12}"
    )
    .fillna(False)
    .sum()
)


upc_leading_zero_count = int(
    upc.str.fullmatch(
        r"0\d{11}"
    )
    .fillna(False)
    .sum()
)


print(
    "SMDUPCA recorded 12-digit UPCs:",
    upc_recorded_count
)

print(
    "SMDUPCA '* NO MATCH *':",
    upc_no_match_count
)

print(
    "SMDUPCA blanks:",
    upc_blank_count
)

print(
    "12-digit UPCs beginning with 0:",
    upc_leading_zero_count
)


if upc_recorded_count != 562:

    raise ValueError(
        "SMDUPCA recorded UPC count differs."
    )


if upc_no_match_count != 106:

    raise ValueError(
        "SMDUPCA NO MATCH count differs."
    )


if upc_blank_count != 6056:

    raise ValueError(
        "SMDUPCA blank count differs."
    )


if upc_leading_zero_count != 514:

    raise ValueError(
        "Unexpected SMDUPCA leading-zero count."
    )


brand_blank_count = int(
    brand.eq("")
    .sum()
)


brand_nonblank_count = int(
    brand.ne("")
    .sum()
)


if brand_blank_count != 5795:

    raise ValueError(
        "Unexpected SMD100BR blank count."
    )


if brand_nonblank_count != 929:

    raise ValueError(
        "Unexpected SMD100BR populated count."
    )


print(
    "PASS: UPC and cigarette-brand character "
    "fields validated."
)


# ============================================================
# 14. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 15. CONVERT ONLY PROVEN INTEGER FIELDS TO Int64
# ============================================================

#
# SMD100NI remains decimal.
#
# SMDUPCA and SMD100BR remain character.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} contains fractional values "
                "and cannot safely become Int64."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 34 proven whole-number fields "
    "converted to Int64."
)

print(
    "PASS: SMD100NI retained as genuine decimal."
)

print(
    "PASS: SMDUPCA and SMD100BR retained as text."
)


# ============================================================
# 16. CDC FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} frequency check."
    )


# ============================================================
# 17. RANGE CHECK HELPER
# ============================================================

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        range_count
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            actual_count
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} range summary."
    )


# ============================================================
# 18. CORE ADULT SMOKING CHECKS
# ============================================================

check_frequency(
    "SMQ020",
    {
        1: 2359,
        2: 3497
    },
    868
)


check_range_summary(
    "SMD030",
    7,
    76,
    2285,
    {
        0: 62,
        777: 0,
        999: 12
    },
    4365
)


check_frequency(
    "SMQ040",
    {
        1: 805,
        2: 216,
        3: 1338
    },
    4365
)


check_range_summary(
    "SMQ050Q",
    1,
    71,
    1255,
    {
        66666: 78,
        77777: 0,
        99999: 5
    },
    5386
)


check_frequency(
    "SMQ050U",
    {
        1: 13,
        2: 11,
        3: 96,
        4: 1135
    },
    5469
)


check_range_summary(
    "SMD057",
    2,
    90,
    1240,
    {
        1: 92,
        95: 1,
        777: 0,
        999: 5
    },
    5386
)


# ============================================================
# 19. CURRENT SMOKING CHECKS
# ============================================================

check_frequency(
    "SMQ078",
    {
        1: 195,
        2: 271,
        3: 162,
        4: 73,
        5: 44,
        6: 13,
        7: 34,
        99: 1
    },
    5931
)


check_range_summary(
    "SMD641",
    0,
    30,
    1060,
    {
        77: 0,
        99: 3
    },
    5661
)


check_range_summary(
    "SMD650",
    2,
    60,
    959,
    {
        1: 60,
        95: 0,
        777: 1,
        999: 2
    },
    5702
)


check_frequency(
    "SMD093",
    {
        1: 672,
        2: 261,
        3: 33,
        4: 55
    },
    5703
)


# ============================================================
# 20. CIGARETTE PRODUCT CHECKS
# ============================================================

check_frequency(
    "SMD100FL",
    {
        0: 15,
        1: 914
    },
    5795
)


check_frequency(
    "SMD100MN",
    {
        0: 553,
        1: 376
    },
    5795
)


check_frequency(
    "SMD100LN",
    {
        1: 60,
        2: 485,
        3: 381,
        4: 3
    },
    5795
)


check_range_summary(
    "SMD100TR",
    6,
    19,
    695,
    {},
    6029
)


check_range_summary(
    "SMD100NI",
    0.5,
    2.0,
    695,
    {},
    6029
)


check_range_summary(
    "SMD100CO",
    1,
    19,
    695,
    {},
    6029
)


# ============================================================
# 21. YOUTH SMOKING CHECK
# ============================================================

check_frequency(
    "SMQ621",
    {
        1: 734,
        2: 44,
        3: 10,
        4: 13,
        5: 7,
        6: 4,
        7: 2,
        8: 6,
        99: 1
    },
    5903
)


# ============================================================
# 22. QUIT ATTEMPT CHECKS
# ============================================================

check_frequency(
    "SMQ670",
    {
        1: 541,
        2: 494
    },
    5689
)


check_range_summary(
    "SMQ848",
    1,
    20,
    519,
    {
        777: 0,
        999: 9
    },
    6196
)


check_range_summary(
    "SMQ852Q",
    1,
    364,
    516,
    {
        0: 3,
        7777: 0,
        9999: 3
    },
    6202
)


check_frequency(
    "SMQ852U",
    {
        1: 258,
        2: 118,
        3: 143
    },
    6205
)


# ============================================================
# 23. CIGAR / E-CIGARETTE / SMOKELESS CHECKS
# ============================================================

check_frequency(
    "SMQ890",
    {
        1: 2095,
        2: 3759,
        9: 2
    },
    868
)


check_range_summary(
    "SMQ895",
    0,
    30,
    2095,
    {
        77: 0,
        99: 0
    },
    4629
)


check_frequency(
    "SMQ900",
    {
        1: 1150,
        2: 4705,
        9: 1
    },
    868
)


check_range_summary(
    "SMQ905",
    0,
    30,
    1149,
    {
        77: 0,
        99: 1
    },
    5574
)


check_frequency(
    "SMQ910",
    {
        1: 861,
        2: 4994,
        9: 1
    },
    868
)


check_range_summary(
    "SMQ915",
    0,
    30,
    861,
    {
        77: 0,
        99: 0
    },
    5863
)


check_frequency(
    "SMAQUEX2",
    {
        1: 5856,
        2: 868
    },
    0
)


# ============================================================
# 24. ROUTING CHECK HELPER
# ============================================================

def check_route(
    description,
    expected_mask,
    child_column
):

    expected = (
        expected_mask
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    present = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )

    mismatch_count = int(
        np.sum(
            expected != present
        )
    )

    print(
        f"\n--- ROUTING: {description} ---"
    )

    print(
        "Expected populated:",
        int(expected.sum())
    )

    print(
        "Actually populated:",
        int(present.sum())
    )

    print(
        "Mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"Routing failed: {description}"
        )

    print(
        "PASS: Routing validated."
    )


# ============================================================
# 25. ADULT CIGARETTE ROUTING
# ============================================================

check_route(
    "SMQ020=1 -> SMD030",
    df["SMQ020"].eq(1),
    "SMD030"
)


check_route(
    "SMQ020=1 -> SMQ040",
    df["SMQ020"].eq(1),
    "SMQ040"
)


check_route(
    "SMQ040=3 -> SMQ050Q",
    df["SMQ040"].eq(3),
    "SMQ050Q"
)


check_route(
    "SMQ040=3 -> SMD057",
    df["SMQ040"].eq(3),
    "SMD057"
)


# ============================================================
# 26. TOBACCO PRODUCT ROUTING
# ============================================================

check_route(
    "SMQ890=1 -> SMQ895",
    df["SMQ890"].eq(1),
    "SMQ895"
)


check_route(
    "SMQ900=1 -> SMQ905",
    df["SMQ900"].eq(1),
    "SMQ905"
)


check_route(
    "SMQ910=1 -> SMQ915",
    df["SMQ910"].eq(1),
    "SMQ915"
)


# ============================================================
# 27. FINAL ZERO PATTERN
# ============================================================

final_zero_counts = {}


for col in NUMERIC_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from validated "
        "SMQ_J correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "validated corrections."
)


# ============================================================
# 28. NUMERIC VALUE FIDELITY BEFORE EXPORT
# ============================================================

print(
    "\n--- NUMERIC VALUE FIDELITY BEFORE EXPORT ---"
)


numeric_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "Unexpected numeric modifications:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 35 numeric variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 29. CHARACTER FIDELITY BEFORE EXPORT
# ============================================================

character_failures = []


for col in CHARACTER_COLUMNS:

    expected_values = (
        source_df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    current_values = (
        df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    if not np.array_equal(
        expected_values,
        current_values
    ):

        character_failures.append(
            col
        )


if character_failures:

    raise ValueError(
        "Character fields changed before export:\n"
        + str(character_failures)
    )


print(
    "PASS: SMDUPCA and SMD100BR unchanged."
)


# ============================================================
# 30. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0


for col in NUMERIC_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Numeric missing positions changed."
    )


print(
    "PASS: Missing positions preserved."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     conversion of SMDUPCA to numeric
#
# NO:
#
#     conversion of SMD100NI to Int64
#
# This protects:
#
#     integer/code fields
#     genuine decimal SMD100NI
#     leading-zero UPC values
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 32. RE-READ NUMERIC CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_numeric = pd.read_csv(
    csv_file,
    usecols=NUMERIC_COLUMNS,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_numeric.shape != (
    6724,
    35
):

    raise ValueError(
        "Numeric CSV structure changed."
    )


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_numeric_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_numeric[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_numeric_failures.append(
            col
        )


if csv_numeric_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_numeric_failures)
    )


print(
    "PASS: All 35 numeric CSV fields "
    "round-trip exactly."
)


# ============================================================
# 34. READ ENTIRE CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    6724,
    37
):

    raise ValueError(
        "Raw CSV structure changed."
    )


if raw_csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 35. FINAL CHARACTER FIDELITY
# ============================================================

print(
    "\n--- FINAL CHARACTER FIDELITY ---"
)


for col in CHARACTER_COLUMNS:

    source_tokens = (
        source_df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    csv_tokens = (
        raw_csv_df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    if not np.array_equal(
        source_tokens,
        csv_tokens
    ):

        raise ValueError(
            f"{col} text changed during CSV export."
        )


print(
    "PASS: Character fields preserved exactly."
)


# ============================================================
# 36. FINAL UPC FIDELITY
# ============================================================

final_upc = (
    raw_csv_df["SMDUPCA"]
    .astype("string")
)


final_upc_recorded = int(
    final_upc.str.fullmatch(
        r"\d{12}"
    )
    .fillna(False)
    .sum()
)


final_upc_nomatch = int(
    final_upc.eq(
        "* NO MATCH *"
    )
    .sum()
)


final_upc_blank = int(
    final_upc.eq("")
    .sum()
)


final_upc_leading_zero = int(
    final_upc.str.fullmatch(
        r"0\d{11}"
    )
    .fillna(False)
    .sum()
)


print("\n--- FINAL UPC CHECK ---")

print(
    "12-digit UPCs:",
    final_upc_recorded
)

print(
    "NO MATCH:",
    final_upc_nomatch
)

print(
    "Blank:",
    final_upc_blank
)

print(
    "Leading-zero UPCs:",
    final_upc_leading_zero
)


if final_upc_recorded != 562:

    raise ValueError(
        "Final UPC recorded count changed."
    )


if final_upc_nomatch != 106:

    raise ValueError(
        "Final UPC NO MATCH count changed."
    )


if final_upc_blank != 6056:

    raise ValueError(
        "Final UPC blank count changed."
    )


if final_upc_leading_zero != 514:

    raise ValueError(
        "Leading-zero UPCs were damaged."
    )


print(
    "PASS: All 514 leading-zero UPCs preserved."
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 38. RAW CSV .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# DO NOT check SMD100NI here.
#
# It is a genuine decimal field.
#

print(
    "\n--- INTEGER-FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' found in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 34 integer/code fields."
)


# ============================================================
# 39. DECIMAL FIELD RAW-TEXT CHECK
# ============================================================

#
# SMD100NI remains a genuine decimal variable.
#
# We deliberately do not force decimal formatting.
#

nicotine_tokens = (
    raw_csv_df["SMD100NI"]
)


nicotine_nonblank = (
    nicotine_tokens[
        nicotine_tokens != ""
    ]
)


print(
    "\n--- SMD100NI RAW DECIMAL CHECK ---"
)

print(
    "Populated decimal tokens:",
    len(nicotine_nonblank)
)

print(
    "Tokens ending in '.0' "
    "(allowed for decimal variable):",
    int(
        nicotine_nonblank
        .str.endswith(".0")
        .sum()
    )
)


if len(nicotine_nonblank) != 695:

    raise ValueError(
        "SMD100NI CSV populated count changed."
    )


#
# Numeric fidelity of SMD100NI was already checked
# using the round-trip parser.
#

print(
    "PASS: SMD100NI remained a genuine "
    "decimal variable."
)


# ============================================================
# 40. FINAL ZERO PATTERN IN CSV
# ============================================================

csv_zero_counts = {}


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    count = int(
        values.eq(0)
        .sum()
    )

    if count > 0:

        csv_zero_counts[col] = count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final zero pattern preserved."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in NUMERIC_COLUMNS:

    tokens = raw_csv_df[col]

    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )

    if count > 0:

        scientific_errors[col] = count


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation in final CSV."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV has duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid and contains no '.0'."
)


# ============================================================
# 43. FINAL NUMERIC SOURCE -> CSV CHECK
# ============================================================

final_numeric_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        final_numeric_failures.append(
            col
        )


if final_numeric_failures:

    raise ValueError(
        "Final numeric fidelity failures:\n"
        + str(final_numeric_failures)
    )


print(
    "PASS: Final CSV exactly preserves all "
    "corrected numeric values."
)


# ============================================================
# 44. FINAL CHARACTER SOURCE -> CSV CHECK
# ============================================================

final_character_failures = []


for col in CHARACTER_COLUMNS:

    source_values = (
        source_df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    final_values = (
        raw_csv_df[col]
        .fillna("")
        .astype("string")
        .to_numpy(
            dtype=str
        )
    )

    if not np.array_equal(
        source_values,
        final_values
    ):

        final_character_failures.append(
            col
        )


if final_character_failures:

    raise ValueError(
        "Final character fidelity failures:\n"
        + str(final_character_failures)
    )


print(
    "PASS: Final CSV exactly preserves "
    "SMDUPCA and SMD100BR."
)


# ============================================================
# 45. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "SMQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - "
    "Smoking / Cigarette Use"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)

print(
    "Tiny-value variables:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "Total zero restorations:",
    sum(
        correction_counts.values()
    )
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Character variables:",
    CHARACTER_COLUMNS
)

print(
    "Whole-number/code numeric variables:",
    len(INTEGER_COLUMNS)
)

print(
    "SMD100NI populated:",
    int(
        df["SMD100NI"]
        .notna()
        .sum()
    )
)

print(
    "SMD100NI range:",
    f"{df['SMD100NI'].min()}-"
    f"{df['SMD100NI'].max()}"
)

print(
    "12-digit UPCs:",
    upc_recorded_count
)

print(
    "Leading-zero UPCs preserved:",
    upc_leading_zero_count
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Character-field differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 SMQ_J XPT -> CSV
Rows: 6,724
Columns: 37
PASS: Structure = 6,724 rows x 37 columns.

--- FIELD TYPE PLAN ---
Character fields: ['SMDUPCA', 'SMD100BR']
Genuine decimal fields: ['SMD100NI']
Whole-number/code numeric fields: 34

--- SOURCE DTYPE CHECK ---
SMDUPCA: dtype = str
SMD100BR: dtype = str
PASS: Character fields loaded as non-numeric text-compatible columns.
PASS: SMDUPCA and SMD100BR explicitly retained as text.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT values: 3,946

Tiny values by variable:
SMQ895      1711
SMQ905       844
SMQ915       720
SMD100MN     553
SMD030        62
SMD641        38
SMD100FL      15
SMQ852Q        3
PASS: Exact 3,946-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
SMD030: 62
SMD641: 38
SMD100FL: 15
SMD100MN: 553
SMQ852Q: 3
SMQ895: 1,711
SMQ905: 844
SMQ915: 720
PASS: 3,946 validated zeros restored.

Tiny XPORT values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- F